# Market Data — Russell 1000 Universe

Fetches daily OHLCV bars from Alpaca for every ticker in `data/ru1000.yml`
and stores them via `MarketDataCache` as:

```
data/historical-prices/<TICKER>/<YEAR>.csv
```

Runs in batches of 50 symbols per API request.  
Already-stored files are skipped so the cell can be re-run safely.

In [6]:
from sentiment.log import setup_logging
setup_logging()

In [7]:
from datetime import datetime
from pathlib import Path

import yaml

from sentiment.sources.alpaca import AlpacaSource
from sentiment.sources.cache import MarketDataCache

In [8]:
with open("../data/ru1000.yml") as f:
    companies = yaml.safe_load(f)["companies"]

tickers = list(companies.keys())

YEARS      = [2018, 2019, 2020]
TIMEFRAME  = "1Day"
BATCH_SIZE = 50   # symbols per Alpaca request

print(f"Tickers   : {len(tickers)}")
print(f"Years     : {YEARS}")
print(f"Batch size: {BATCH_SIZE}")

Tickers   : 992
Years     : [2018, 2019, 2020]
Batch size: 50


In [9]:
alpaca = AlpacaSource()
cache  = MarketDataCache()

for year in YEARS:
    start = datetime(year, 1, 1)
    end   = datetime(year, 12, 31)
    n_batches = (len(tickers) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\n=== {year} ({start.date()} \u2192 {end.date()}) — {n_batches} batches ===")

    stored  = 0
    skipped = 0
    errors  = 0

    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i : i + BATCH_SIZE]

        # Skip tickers whose <TICKER>/<YEAR>.csv already exists
        to_fetch = [t for t in batch if not cache._path(t, year).exists()]
        skipped += len(batch) - len(to_fetch)

        if not to_fetch:
            continue

        try:
            df = alpaca.fetch_bars(to_fetch, TIMEFRAME, start, end)
        except Exception as exc:
            print(f"  [ERROR] batch {i // BATCH_SIZE + 1}: {exc}")
            errors += len(to_fetch)
            continue

        if df.empty:
            continue

        for ticker, group in df.groupby("symbol"):
            cache.store(ticker, year, group)
            stored += 1

        batch_num = i // BATCH_SIZE + 1
        print(f"  batch {batch_num:3d}/{n_batches}  stored={stored}  skipped={skipped}  errors={errors}")

    print(f"Year {year} complete — stored: {stored}, skipped: {skipped}, errors: {errors}")

print("\nAll years complete.")


=== 2018 (2018-01-01 → 2018-12-31) — 20 batches ===
  batch   1/20  stored=42  skipped=0  errors=0
  batch   2/20  stored=85  skipped=0  errors=0
  batch   3/20  stored=129  skipped=0  errors=0
  batch   4/20  stored=174  skipped=0  errors=0
  batch   5/20  stored=216  skipped=0  errors=0
  batch   6/20  stored=259  skipped=0  errors=0
  batch   7/20  stored=302  skipped=0  errors=0
  batch   8/20  stored=349  skipped=0  errors=0
  batch   9/20  stored=392  skipped=0  errors=0
  batch  10/20  stored=435  skipped=0  errors=0
  batch  11/20  stored=479  skipped=0  errors=0
  batch  12/20  stored=522  skipped=0  errors=0
  batch  13/20  stored=567  skipped=0  errors=0
  batch  14/20  stored=609  skipped=0  errors=0
  batch  15/20  stored=653  skipped=0  errors=0
  batch  16/20  stored=697  skipped=0  errors=0
  batch  17/20  stored=742  skipped=0  errors=0
  batch  18/20  stored=788  skipped=0  errors=0
  batch  19/20  stored=833  skipped=0  errors=0
  batch  20/20  stored=873  skipped=0